In [0]:
# ============================================================
# Lab 1 / Phase 1 / Stage A — Cash Flow Gold Layer
# Source: bdc_share_cash_flow.cashflow.cashflow (SAP BDC Delta Share)
# ============================================================

# 1. Create a dedicated schema for Lab 1 outputs (keeps it separate)
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.sap_finance_agent")

# 2. Build gold table: monthly cash flow by company code
spark.sql("""
CREATE OR REPLACE TABLE workspace.sap_finance_agent.gold_monthly_cashflow AS
SELECT
    CompanyCode                                    AS company_code,
    DATE_TRUNC('MONTH', TransactionDate)           AS cashflow_month,
    CompanyCodeCurrency                            AS currency,
    COUNT(*)                                       AS transaction_count,
    SUM(AmountInCompanyCodeCurrency)               AS net_cashflow,
    SUM(CASE WHEN AmountInCompanyCodeCurrency > 0
             THEN AmountInCompanyCodeCurrency ELSE 0 END) AS total_inflow,
    SUM(CASE WHEN AmountInCompanyCodeCurrency < 0
             THEN AmountInCompanyCodeCurrency ELSE 0 END) AS total_outflow
FROM bdc_share_cash_flow.cashflow.cashflow
WHERE TransactionDate IS NOT NULL
  AND CompanyCode IS NOT NULL
GROUP BY CompanyCode, DATE_TRUNC('MONTH', TransactionDate), CompanyCodeCurrency
""")

# 3. Verify
df = spark.sql("SELECT * FROM workspace.sap_finance_agent.gold_monthly_cashflow ORDER BY company_code, cashflow_month")
print("Gold table rows:", df.count())
display(df)